# Practical 3

## Age prediction

In [1]:
%%bash
datalad get --jobs 4 ds000030-halfpipe

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "nilearn==0.13.1", "scikit-learn==1.9.1"],
    check=False,
)

CompletedProcess(args=['/opt/conda/bin/python', '-m', 'pip', 'install', 'nilearn==0.13.1'], returncode=0)

In [3]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from numpy import typing as npt


@dataclass
class TimeSeries:
    """
    Represents a time series.

    Attributes:
        path (Path): The path to the ".tsv" file containing the time series.
        metadata (dict[str, Any]): Additional metadata associated with the connectivity matrix.
    """

    path: Path
    confounds_path: Path

    def __post_init__(self) -> None:
        if not self.path.is_file() and self.confounds_path.is_file():
            raise ValueError("Files not found")
    
    @property
    def subject(self) -> str:
        return self.path.name.split("_")[0]
    
    @property
    def values(self) -> npt.NDArray[np.float64]:
        """
        Time series values from the file.

        Returns:
            ndarray: The time series as a NumPy array.
        """
        return np.loadtxt(self.path, delimiter="\t")
    
    @property
    def confounds(self) -> pd.DataFrame:
        """
        Associated confound time series.

        Returns:
            ndarray: The confounds time series as a Pandas data frame.
        """
        return pd.read_table(self.confounds_path)

In [4]:
time_series = [
    TimeSeries(
        path=path,
        confounds_path=path.parent / path.name.replace("timeseries", "desc-confounds_timeseries"),
    )
    for path in Path("ds000030-halfpipe").rglob("*_task-rest_timeseries.tsv")
]

In [5]:
data_frame = pd.read_table("ds000030-halfpipe/participants.tsv").set_index("participant_id").loc[(time_series.subject for time_series in time_series)]
y = pd.Series(data_frame.age)

In [ ]:
from nilearn.connectome import sym_matrix_to_vec
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

confounds_columns = ["white_matter", "csf"]

def calculate_correlation_matrix(time_series: TimeSeries) -> npt.NDArray[np.float64]:
    confounds = time_series.confounds[confounds_columns].to_numpy()
    values = np.nan_to_num(time_series.values)
    
    model = make_pipeline(
        SimpleImputer(strategy='constant'),
        StandardScaler(),
        LinearRegression()
    )
    
    predicted_values = model.fit(confounds, values).predict(confounds)
    residuals = values - predicted_values
    return pd.DataFrame(residuals).corr().to_numpy()

correlation_matrices = np.asarray([
    calculate_correlation_matrix(time_series) for time_series in tqdm(time_series)
])
    
X = sym_matrix_to_vec(correlation_matrices, discard_diagonal=True)

  0%|          | 0/194 [00:00<?, ?it/s]

In [ ]:
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

model = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("variance_threshold", VarianceThreshold(threshold=1e-6)),
        ("scaler", StandardScaler()),
        (
            "pca",
            PCA(
                n_components=100,
                svd_solver="randomized",
                random_state=0,
            ),
        ),
        ("estimator", Ridge(alpha=1.0)),
    ]
)

result = cross_validate(
    model, X, y, scoring=("neg_mean_absolute_error", "r2"), verbose=3, n_jobs=4
)

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   2 out of   5 | elapsed:   10.0s remaining:   15.0s
[Parallel(n_jobs=4)]: Done   5 out of   5 | elapsed:   19.1s finished


In [10]:
result

{'fit_time': array([ 9.95459175, 10.01699114,  9.51454997,  9.62790108,  9.01476049]),
 'score_time': array([0.17485857, 0.1791985 , 0.17717385, 0.1746614 , 0.17429233]),
 'test_neg_mean_absolute_error': array([-7.63309367, -7.07652725, -6.02868253, -6.80478201, -8.12432001]),
 'test_r2': array([0.04620534, 0.05089471, 0.36377259, 0.31724266, 0.03276833])}

In [ ]:
import joblib

joblib.dump(
    {
        "model": model,
        "confounds_columns": confounds_columns,
    },
    "model.joblib",
)